In [1]:
import os

In [2]:
%pwd

'c:\\Users\\sam\\End-to-End-Fashion-Recommendation-System-with-MLOps\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'c:\\Users\\sam\\End-to-End-Fashion-Recommendation-System-with-MLOps'

In [5]:
from src.recommendation_system.constants import CONFIG_PATH
from src.recommendation_system.utils.common import read_yaml , create_dir
from dataclasses import dataclass
from src.recommendation_system.logging import logger
import pickle

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class model_building_config:
    sim_matrix_path:  Path    
    featured_df_path: Path    
    model_path:       Path    


In [ ]:
class Config_manager:

    def __init__(self, config =CONFIG_PATH):
        self.config = read_yaml(config)

        create_dir([self.config.artifacts_root])

    def get_model_building(self) -> model_building_config:
        
        config = self.config.model_building

        create_dir([config.model_path])

        return   model_building_config(
            sim_matrix_path  = (config.sim_matrix_path),
            featured_df_path = (config.featured_df_path),
            model_path       = (config.model_path)
        )
               

In [8]:
class model_building_config:

    def __init__(self, config) -> None:
        self.config = config
        self.sim_matrix = None
        self.df = None

    def load_artifacts(self):
        try:
            logger.info("=" * 20 + " Loading Artifacts STARTED " + "=" * 20)

            with open(self.config.sim_matrix_path, 'rb') as f:
                self.sim_matrix = pickle.load(f)
            with open(self.config.featured_df_path, 'rb') as f:
                self.df = pickle.load(f)

            logger.info("sim_matrix shape : %s", str(self.sim_matrix.shape))
            logger.info("df shape         : %s", str(self.df.shape))
            logger.info("df columns       : %s", list(self.df.columns))
            logger.info("=" * 20 + " Loading Artifacts COMPLETED " + "=" * 20)

            return self.sim_matrix, self.df

        except Exception as e:
            logger.error("Load artifacts FAILED - %s", str(e))
            raise e

    def recommendation_engine_1(self, product_id, n=10):
        # similarity based recommendations
        try:
            scores  = self.sim_matrix[product_id]
            top_idx = scores.argsort()[::-1][1:n+1]
            top_scores = scores[top_idx]

            results = self.df.iloc[top_idx][[
                'product_name_clean',
                'price',
                'rating',
                'aesthetic',
                'score',
            ]].copy()

            results['similarity'] = top_scores

            logger.info("[Engine 1] Top %d similar products found", n)
            logger.info("=" * 20 + " Engine 1 COMPLETED " + "=" * 20)

            return results, top_idx

        except Exception as e:
            logger.error("[Engine 1] FAILED - %s", str(e))
            raise e

    def recommendation_engine_2(self, aesthetic, n=10):
        # aesthetic/category based recommendations
        try:
            filtered = self.df[self.df['aesthetic'] == aesthetic]

            top_products = filtered.sort_values(
                'score', ascending=False
            ).head(n)

            logger.info("[Engine 2] Total in aesthetic : %d", len(filtered))
            logger.info("[Engine 2] Top %d products found", n)
            logger.info("=" * 20 + " Engine 2 COMPLETED " + "=" * 20)

            return top_products

        except Exception as e:
            logger.error("[Engine 2] FAILED - %s", str(e))
            raise e

    def initiate_recommendation(self, product_name, top_k=5):
        try:
            logger.info("=" * 20 + " Recommendation STARTED " + "=" * 20)

            # load artifacts
            self.sim_matrix, self.df = self.load_artifacts()

            # validate product exists
            if product_name not in self.df["product_name_clean"].values:
                raise ValueError(f"Product '{product_name}' not found")

            # get product_id and aesthetic
            product_id = self.df[self.df["product_name_clean"] == product_name].index[0]
            aesthetic  = self.df.loc[product_id, "aesthetic"]

            logger.info("Input product : %s", product_name)
            logger.info("Product ID    : %d", product_id)
            logger.info("Aesthetic     : %s", aesthetic)

            # engine 1 — similarity based
            recs_sim, top_idx = self.recommendation_engine_1(
                product_id=product_id,
                n=top_k
            )

            # engine 2 — aesthetic based
            recs_aesthetic = self.recommendation_engine_2(
                aesthetic=aesthetic,
                n=top_k
            )

            logger.info("=" * 20 + " Recommendation COMPLETED " + "=" * 20)

            return recs_sim, recs_aesthetic

        except Exception as e:
            logger.error("Recommendation FAILED - %s", str(e))
            raise e

In [9]:
con = Congif_manager()
model_feature = con.get_model_building()
model_feature = model_building_config(model_feature)
df = model_feature.initiate_recommendation('men wonder sports running shoes',top_k=10)


[2026-03-06 18:53:57,807: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-06 18:53:57,811: INFO: common: created directory at: artifacts]
[2026-03-06 18:53:57,813: INFO: common: created directory at: artifacts/model_building/model/]
Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "c:\Users\sam\End-to-End-Fashion-Recommendation-System-with-MLOps\venv\Lib\site-packages\IPython\core\interactiveshell.py", line 3701, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\sam\AppData\Local\Temp\ipykernel_20844\1841276756.py", line 2, in <module>
    model_feature = con.get_model_building()
                    ^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\sam\AppData\Local\Temp\ipykernel_20844\342901110.py", line 14, in get_model_building
    return   model_building_config(
             ^^^^^^^^^^^^^^^^^^^^^^
TypeError: model_building_config.__init__() got an unexpected keyword argument 'sim_matrix_path'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\sam\End-to-End-Fashion-Recommendation-System-with-MLOps\venv\Lib\site-packages\IPython\core\interactiveshell.py", line 2201, in showtraceback
    stb = self.InteractiveTB.structured_

In [10]:
import inspect
from src.recommendation_system.entity import model_building_config

print("File:", inspect.getfile(model_building_config))
print("Fields:", [f.name for f in model_building_config.__dataclass_fields__.values()])

File: c:\Users\sam\End-to-End-Fashion-Recommendation-System-with-MLOps\src\recommendation_system\entity\__init__.py
Fields: ['sim_matrix_path', 'featured_df_path', 'model_path']


In [13]:
import inspect
from src.recommendation_system.config.config import Config_manager
from src.recommendation_system.entity import model_building_config

# Check both are from the same place
print("Config file:", inspect.getfile(Config_manager))
print("Entity file:", inspect.getfile(model_building_config))

# Check what config.py thinks model_building_config is
import src.recommendation_system.config.config as cfg_module
print("Config module's model_building_config:", inspect.getfile(cfg_module.model_building_config))

Config file: c:\Users\sam\End-to-End-Fashion-Recommendation-System-with-MLOps\src\recommendation_system\config\config.py
Entity file: c:\Users\sam\End-to-End-Fashion-Recommendation-System-with-MLOps\src\recommendation_system\entity\__init__.py
Config module's model_building_config: c:\Users\sam\End-to-End-Fashion-Recommendation-System-with-MLOps\src\recommendation_system\entity\__init__.py


In [14]:
import inspect
from src.recommendation_system.entity import model_building_config

print(inspect.getsource(model_building_config))

@dataclass(frozen=True)
class model_building_config:
    sim_matrix_path:  Path    
    featured_df_path: Path    
    model_path:       Path    

